# db_10: Welches Modell ist am besten?

**Frage:** Welches Modell sagt aus den Excel-Daten die 3-Tage-Richtung des Heizöl- und Dieselpreises am besten voraus, und ist das Ergebnis wirklich besser als eine einfache Regel?

**Fairer Vergleich:** Alle Modelle bekommen dieselben Daten und dieselben Merkmale. Sie werden jeden Tag neu mit den dann bekannten Daten trainiert (kein Blick in die Zukunft). Dasselbe Szenario wie vorher (S0: Kauf zum Preis von Tag t).

**Modelle:** XGBoost (bisher), Logistische Regression, Random Forest, Logistisch klein (nur 7 Merkmale), Mehrheitsklasse (Dummy: tippt immer die häufigere Richtung).
**Einfache Regeln zum Vergleich:** NYMEX-Regel (US-Preis gestiegen = Kaufen) und Momentum (gestern gestiegen = Kaufen).

**Zwei Zeiträume:** Entwicklung (2023-04-21 bis 2024-11-01) und Test (2024-11-15 bis 2026-01-20). Die Rangfolge gilt nur dann als belastbar, wenn sie in **beiden** Zeiträumen ähnlich ist.

**Ehrlicher Vorbehalt vorab:** Mit etwa 300 Tagen und überlappenden 3-Tage-Labels sind kleine Unterschiede zwischen Modellen Zufall. Ein klarer Sieger ist nur zu erwarten, wenn er deutlich vorne liegt.

Ablauf: Installation, Zelle 0 bis 7 der Reihe nach ausführen. Zelle 4 dauert lang (bis zu 1 Stunde, der Random Forest ist der langsamste), sie speichert Zwischenstände und kann nach einem Abbruch einfach neu gestartet werden.

In [ ]:
%pip install xgboost scikit-learn openpyxl

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Zelle 0: Legt den Projektcode (Ordner p229) auf dem Cluster an. Nichts ändern, nur ausführen.
import os
os.makedirs("/tmp/p229_code/p229", exist_ok=True)
FILES = {}
FILES['__init__'] = r'''"""P229: Kauf/Warten-Signal für Heizöl und Diesel (BayWa). Logik aus db_08_xgboost_sauber."""
'''
FILES['config'] = r'''"""Konstanten aus db_08 (unverändert übernommen)."""
import pandas as pd

SERIES = ["heizoel_sued", "heizoel_suedwest", "heizoel_suedost", "heizoel_rheinmain",
          "diesel_sued", "diesel_suedwest", "diesel_suedost", "diesel_rheinmain"]
EXO = ["ice_gasoil", "brent", "wti", "nymex_heating_oil", "usd_eur"]
LAST_ARGUS = pd.Timestamp("2026-01-20")   # danach OMR + fortgeschriebener ICE-Preis
DEV = (pd.Timestamp("2023-04-21"), pd.Timestamp("2024-11-01"))
TEST = (pd.Timestamp("2024-11-15"), pd.Timestamp("2026-01-20"))
HORIZON = 3
REG = dict(max_depth=2, n_estimators=150, learning_rate=0.03, min_child_weight=10,
           reg_lambda=10, subsample=0.7, colsample_bytree=0.5,
           eval_metric="logloss", random_state=42)
# Szenario -> (entry_lag, feat_lag)
SCEN = {"S0 (Kauf zu Preis t)": (0, 0),
        "S1 (Kauf zu Preis t+1)": (1, 0),
        "S2 (Daten nur bis t-1)": (0, 1)}
'''
FILES['stats'] = r'''from math import sqrt

import numpy as np
from scipy.stats import binomtest


def wil(k, n, z=1.96):
    """Wilson-95-%-Intervall für eine Trefferquote k/n."""
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h


def block_boot_mean(x, block=5, n_boot=2000, seed=42):
    """95-%-Intervall des Mittelwerts per Block-Bootstrap (berücksichtigt überlappende Labels)."""
    x = np.asarray(x, float)
    n = len(x)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    starts = np.arange(n - block + 1)
    means = np.empty(n_boot)
    for b in range(n_boot):
        s = rng.choice(starts, nb)
        means[b] = x[(s[:, None] + np.arange(block)).ravel()[:n]].mean()
    return np.percentile(means, [2.5, 97.5])


def mcnemar(model_ok, other_ok):
    """Exakter McNemar-Test (gepaarte Treffer, zweiseitig)."""
    oa, ob = int((model_ok & ~other_ok).sum()), int((~model_ok & other_ok).sum())
    return binomtest(oa, oa + ob, 0.5).pvalue if oa + ob else 1.0


def holm(p):
    """Holm-Bonferroni-adjustierte p-Werte."""
    p = np.asarray(p, float)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[idx])
        adj[idx] = min(1.0, run)
    return adj
'''
FILES['features'] = r'''import numpy as np
import pandas as pd

from .config import EXO


def build_features(df, H, exo=EXO):
    """Stationäre Merkmale (Log-Returns, Nachlauf-Lücke, Volatilität, Wochentag) für Serie H.

    Jedes Merkmal in Zeile t nutzt nur Preise bis einschließlich t.
    """
    df = df.reset_index(drop=True)
    F = pd.DataFrame({"date": df["date"]})
    lp = {c: np.log(df[c].where(df[c] > 0)) for c in [H] + list(exo)}
    for c, s in lp.items():
        r1 = s.diff()
        F[f"{c}_r1"] = r1
        F[f"{c}_r1_l1"] = r1.shift(1)
        F[f"{c}_r1_l2"] = r1.shift(2)
        F[f"{c}_r3"] = s - s.shift(3)
        F[f"{c}_r5"] = s - s.shift(5)
    for c in exo:                                   # Nachlauf-Lücke: Marktbewegung minus Preisbewegung
        F[f"gap1_{c}"] = F[f"{c}_r1"] - F[f"{H}_r1"]
        F[f"gap3_{c}"] = F[f"{c}_r3"] - F[f"{H}_r3"]
    r1h = F[f"{H}_r1"]
    F["vol5_H"] = r1h.rolling(5).std()
    F["vol20_H"] = r1h.rolling(20).std()
    F["dev20_H"] = lp[H] - lp[H].rolling(20).mean()
    F["weekday"] = df["date"].dt.weekday
    return F


def london_features(H, F):
    """Argus-only-Variante: eigene Serie + ICE Gasoil + EUR/USD."""
    keep = ["date", "weekday", "vol5_H", "vol20_H", "dev20_H"]
    for c in [H, "ice_gasoil", "usd_eur"]:
        keep += [f"{c}_r1", f"{c}_r1_l1", f"{c}_r1_l2", f"{c}_r3", f"{c}_r5"]
    for c in ["ice_gasoil", "usd_eur"]:
        keep += [f"gap1_{c}", f"gap3_{c}"]
    return F[keep]
'''
FILES['leakage'] = r'''import numpy as np

from .features import build_features


def leak_test(d, H, n_cut=1500, n_check=50):
    """Features dürfen sich nicht ändern, wenn man die Zukunft abschneidet.

    Braucht mindestens n_cut Zeilen. Wirft AssertionError bei Look-ahead.
    """
    if len(d) < n_cut:
        raise ValueError(f"leak_test braucht >= {n_cut} Zeilen, d hat {len(d)}")
    a = build_features(d.iloc[:n_cut], H).iloc[n_cut - n_check:n_cut].drop(columns="date").values
    b = build_features(d, H).iloc[n_cut - n_check:n_cut].drop(columns="date").values
    assert np.allclose(a, b, equal_nan=True), f"Look-ahead in den Features ({H})!"
'''
FILES['walk_forward'] = r'''import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from .config import HORIZON, REG


def walk_forward(d, H, F, window, entry_lag=0, feat_lag=0, retrain_every=5, params=REG, horizon=HORIZON,
                 model_factory=None):
    """Gepurgter Walk-Forward (expanding window).

    d: Datentabelle (Spalte date + Preisspalten), F: Features zu d, window: (start, ende).
    Ziel: Preis(t+horizon) > Preis(t+entry_lag). Features: Information bis t-feat_lag.
    Training nur mit Zeilen, deren Label am Entscheidungstag i schon bekannt ist:
    last = i - horizon - feat_lag.
    model_factory: Funktion ohne Argumente, die ein frisches Modell (fit/predict_proba) liefert.
    Ohne Angabe: XGBClassifier(**params) wie in db_08.
    """
    feats = [c for c in F.columns if c != "date"]
    P = d[H].astype(float)
    p0, p1 = P.shift(-entry_lag), P.shift(-horizon)
    y = (p1 > p0).astype(float).where(p0.notna() & p1.notna())
    X = F[feats].shift(feat_lag).values.astype(float)
    yv = y.values
    idx = np.where((d["date"] >= window[0]) & (d["date"] <= window[1]))[0]
    proba, model = [], None
    for k, i in enumerate(idx):
        if model is None or k % retrain_every == 0:
            last = i - horizon - feat_lag
            Xt, yt = X[:last + 1], yv[:last + 1]
            m = ~np.isnan(yt)
            model = (model_factory() if model_factory else XGBClassifier(**params)).fit(Xt[m], yt[m])
        proba.append(model.predict_proba(X[i:i + 1])[0, 1])
    res = pd.DataFrame({"date": d["date"].values[idx], "y_true": yv[idx], "proba": proba})
    return res.dropna(subset=["y_true"]).reset_index(drop=True)
'''
FILES['evaluate'] = r'''import numpy as np
import pandas as pd
from scipy.stats import binomtest

from .config import HORIZON
from .stats import block_boot_mean, mcnemar, wil


def evaluate_run(d, H, F, res, entry_lag, feat_lag, horizon=HORIZON):
    """Pflichtmetriken für einen Walk-Forward-Lauf (Logik 1:1 aus db_08).

    F muss das volle Merkmalsset enthalten (für die Baselines NYMEX-Regel und Momentum).
    """
    y, p = res["y_true"].values, res["proba"].values
    pred = (p >= 0.5).astype(int)
    ok = (pred == y)
    n = len(res)
    hit = int(ok.sum())
    lo, hi = wil(hit, n)
    blo, bhi = block_boot_mean(ok.astype(float))
    p_non = max(binomtest(int(ok[o::3].sum()), len(ok[o::3]), 0.5).pvalue for o in range(3))
    base = y.mean()
    brier, ref = np.mean((p - y) ** 2), np.mean((base - y) ** 2)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    # Vorzeichenregeln als Baselines (gleiche Tage)
    pos = pd.Series(np.arange(len(d)), index=d["date"])
    i = pos.reindex(res["date"]).values.astype(int)
    nym = (F["nymex_heating_oil_r1"].shift(feat_lag).values[i] > 0).astype(int)
    mom = (F[f"{H}_r1"].shift(feat_lag).values[i] > 0).astype(int)
    # Kosten (Regret): kaufen = Preis(t+entry_lag), warten = Preis(t+horizon)
    p_now, p_fut = d[H].values[i + entry_lag], d[H].values[i + horizon]
    best = np.minimum(p_now, p_fut)
    c_mod = np.where(pred == 1, p_now, p_fut) - best
    c_buy = p_now - best
    diff = c_buy - c_mod                                   # > 0: Modell billiger als immer kaufen
    dlo, dhi = block_boot_mean(diff)
    return {"Serie": H, "n": n, "DA": hit / n, "Wilson_lo": lo, "Wilson_hi": hi,
            "Boot_lo": blo, "Boot_hi": bhi, "p_nicht_ueberl": p_non,
            "Mehrheit": max(base, 1 - base), "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1),
            "Brier_Skill": 1 - brier / ref,
            "DA_NYMEX_Regel": float((nym == y).mean()), "DA_Momentum": float((mom == y).mean()),
            "p_McNemar_NYMEX": mcnemar(ok, nym == y), "p_McNemar_Momentum": mcnemar(ok, mom == y),
            "Regret_Modell": c_mod.mean(), "Regret_immer_kaufen": c_buy.mean(),
            "Ersparnis_vs_kaufen": diff.mean(), "Ersp_lo": dlo, "Ersp_hi": dhi,
            "davon_zu_frueh_gekauft": c_mod[pred == 1].sum() / n,
            "davon_zu_lange_gewartet": c_mod[pred == 0].sum() / n}
'''
FILES['data'] = r'''"""Daten aus den Excel-Dateien laden und zu einer Tabelle `d` zusammenführen.

Argus-Excel (Add-in): Blatt "Tabelle1", Zeile 2 = Beschreibung, ab Zeile 3 Tageswerte.
Marktdaten-Excel: je Blatt Datum + Wert (Brent, WTI, Heizöl/NYMEX, Wechselkurs).
"""
import warnings

import numpy as np
import pandas as pd

from .config import EXO, SERIES

_REGION = [("southwest", "suedwest"), ("southeast", "suedost"), ("rhine-main", "rheinmain"), ("south", "sued")]


def _serie_name(desc):
    """Beschreibungstext der Argus-Spalte -> Serienname (oder None, z. B. 'all regions')."""
    t = str(desc).lower()
    if "ice nwe month 1" in t:
        return "ice_gasoil"
    if "all regions" in t:
        return None
    produkt = "diesel" if "diesel en 590" in t else "heizoel" if "heating oil" in t else None
    if produkt is None:
        return None
    for key, name in _REGION:
        if key in t:
            return f"{produkt}_{name}"
    return None


def load_argus_excel(path, sheet="Tabelle1"):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    head = raw.iloc[1]                                   # Zeile 2: Beschreibungen
    body = raw.iloc[2:].copy()
    cols = {0: "date"}
    for j in range(1, raw.shape[1]):
        n = _serie_name(head.iloc[j])
        if n:
            cols[j] = n
    df = body[list(cols)].rename(columns=cols)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date")
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    fehlt = [c for c in SERIES + ["ice_gasoil"] if c not in df.columns]
    if fehlt:
        raise ValueError(f"Argus-Spalten nicht erkannt: {fehlt}")
    return df.drop_duplicates("date", keep="last").reset_index(drop=True)


def inspect_excel(path, n=6):
    """Struktur jeder Tabelle ausgeben (Blattnamen, Größe, erste Zeilen) – zum Prüfen vor dem Laden."""
    xl = pd.ExcelFile(path)
    for s in xl.sheet_names:
        df = xl.parse(s, header=None, nrows=n + 4)
        print(f"--- Blatt '{s}' (erste Zeilen) ---")
        print(df.head(n + 4).to_string(max_colwidth=40))


def load_market_sheet(path, sheet, name, date_col=None, value_col=None, header_row=None):
    """Ein Marktdaten-Blatt als Serie `name`.

    Ohne Angabe wird das Blatt automatisch gelesen: erste Spalte mit Datumswerten, danach die
    erste Spalte mit überwiegend Zahlen. Bei Zweifel date_col/value_col (0-basiert) und
    header_row angeben, nachdem `inspect_excel` die Struktur gezeigt hat.
    """
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dt = raw.apply(lambda s: pd.to_datetime(s, errors="coerce"))
    if date_col is None:
        is_date = [(dt[c].notna() & pd.to_numeric(raw[c], errors="coerce").isna()).mean() for c in raw.columns]
        date_col = int(np.argmax(is_date))
        if max(is_date) < 0.5:
            raise ValueError(f"Blatt '{sheet}': keine Datumsspalte gefunden")
    if value_col is None:
        num = [(pd.to_numeric(raw[c], errors="coerce").notna().mean() if c != date_col else -1) for c in raw.columns]
        value_col = int(np.argmax(num))
    out = pd.DataFrame({"date": dt[date_col], name: pd.to_numeric(raw[value_col], errors="coerce")})
    out = out.dropna(subset=["date", name])
    out = out[out["date"] >= "1990-01-01"].drop_duplicates("date", keep="last").sort_values("date")
    return out.reset_index(drop=True)


def build_dataset(argus, markt, max_ffill=5):
    """Argus-Handelstage als Gerüst, Marktdaten per Datum dazu, Lücken bis `max_ffill` Tage vorwärts füllen.

    Rückgabe: (d, bericht). Das Vorwärtsfüllen nutzt nur vergangene Werte (kein Look-ahead).
    """
    d = argus.copy()
    for m in markt:
        d = d.merge(m, on="date", how="left")
    cols = SERIES + EXO
    fehlt = [c for c in cols if c not in d.columns]
    if fehlt:
        raise ValueError(f"Spalten fehlen: {fehlt}")
    vorher = d[cols].isna().sum()
    d[cols] = d[cols].ffill(limit=max_ffill)
    d = d.dropna(subset=cols).reset_index(drop=True)
    bericht = pd.DataFrame({"fehlend_vor_ffill": vorher, "fehlend_nach_ffill": d[cols].isna().sum()})
    return d, bericht
'''
FILES['models'] = r'''"""Modelle für den Vergleich (db_10). Alle bekommen dieselben Daten, denselben Walk-Forward.

Imputation (Median) und Skalierung sind Teil des Modells und werden bei jedem Training nur auf den
Trainingsdaten gelernt (kein Look-ahead).
"""
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from .config import REG


def make_xgb():
    return XGBClassifier(**REG)


def make_logreg():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=0.05, max_iter=1000))


def make_rf():
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=20,
                                                max_features="sqrt", random_state=42, n_jobs=-1))


def make_majority():
    return DummyClassifier(strategy="prior")          # immer die im Training häufigere Klasse


def klein_features(H):
    """Kleines Modell: nur die wichtigsten Tagesrenditen und Nachlauf-Lücken."""
    return [f"{H}_r1", "ice_gasoil_r1", "nymex_heating_oil_r1", "brent_r1", "wti_r1",
            "gap1_ice_gasoil", "gap1_nymex_heating_oil"]


# Name -> (Fabrik, Merkmale: None = alle 44, "klein" = kleine Auswahl)
MODELLE = {
    "XGBoost": (make_xgb, None),
    "Logistische Regression": (make_logreg, None),
    "Random Forest": (make_rf, None),
    "Logistisch klein (7 Merkmale)": (make_logreg, "klein"),
    "Mehrheitsklasse": (make_majority, None),
}
'''
for name, text in FILES.items():
    open(f"/tmp/p229_code/p229/{name}.py", "w").write(text)
print("Code angelegt:", sorted(os.listdir("/tmp/p229_code/p229")))

In [ ]:
# Zelle 1: Pfade, Code, Daten. (Zelle 0, die den Code anlegt, steht direkt über dieser Zelle.)
import os, sys, time
import numpy as np, pandas as pd

BASE_PATH = "/Volumes/dev_workspace/p229/p229_files"
OUT = f"{BASE_PATH}/output"
ARGUS_XLSX = f"{BASE_PATH}/Heizoel und Diesel bis 2016 und ICE Daten.xlsx"
MARKT_XLSX = f"{BASE_PATH}/P229 - Daten von Brent, WTI, Heizöl, Wechselkurs.xlsx"

sys.path.insert(0, "/tmp/p229_code")
from p229.config import SERIES, EXO, DEV, TEST, LAST_ARGUS
from p229.data import load_argus_excel, load_market_sheet, build_dataset
from p229.features import build_features
from p229.leakage import leak_test
from p229.walk_forward import walk_forward
from p229.evaluate import evaluate_run
from p229.models import MODELLE, klein_features
from p229.stats import holm, mcnemar

argus = load_argus_excel(ARGUS_XLSX)
markt = [
    load_market_sheet(MARKT_XLSX, "Brent Rohöl", "brent"),
    load_market_sheet(MARKT_XLSX, "WIT Rohöl", "wti"),
    load_market_sheet(MARKT_XLSX, "Heizöl", "nymex_heating_oil"),
    load_market_sheet(MARKT_XLSX, "USD_EUR Wechselkurs", "usd_eur"),
]
d_all, _ = build_dataset(argus, markt)
d = d_all[d_all["date"] <= LAST_ARGUS].reset_index(drop=True)
print("Tabelle d:", d.shape, d["date"].min().date(), "bis", d["date"].max().date())
for H in SERIES:
    leak_test(d, H)
print("Leakage-Test bestanden")

In [ ]:
# Zelle 2: Welche Serien und Modelle laufen? Für einen schnellen Probelauf (ca. 10 Minuten)
# hier zwei Serien eintragen, z. B. SERIEN_LAUF = ["heizoel_sued", "diesel_sued"].
SERIEN_LAUF = list(SERIES)
FENSTER = {"Entwicklung": DEV, "Test": TEST}
SZENARIO = (0, 0)          # S0: Kauf zu Preis t, Daten bis t
print("Serien:", SERIEN_LAUF)
print("Modelle:", list(MODELLE))

In [ ]:
# Zelle 3: Hilfsfunktion. Jeder Lauf wird als CSV gesichert und beim Neustart wiederverwendet.
def lauf(modell, H, fenster):
    fabrik, modus = MODELLE[modell]
    F_voll = build_features(d, H)
    F_mod = F_voll if modus is None else F_voll[["date"] + klein_features(H)]
    sicher = modell.replace(" ", "_").replace("(", "").replace(")", "")
    pfad = f"{OUT}/db10_proba_{sicher}_{H}_{fenster}.csv"
    if os.path.exists(pfad):
        res = pd.read_csv(pfad, parse_dates=["date"])
    else:
        res = walk_forward(d, H, F_mod, FENSTER[fenster], entry_lag=SZENARIO[0], feat_lag=SZENARIO[1],
                           model_factory=fabrik)
        res.to_csv(pfad, index=False)
    return res, evaluate_run(d, H, F_voll, res, SZENARIO[0], SZENARIO[1])

In [ ]:
# Zelle 4: Alle Modelle, alle Serien, beide Zeiträume (bis zu 1 Stunde für alles).
rows, RES = [], {}
t0 = time.time()
for modell in MODELLE:
    for H in SERIEN_LAUF:
        for fenster in FENSTER:
            res, r = lauf(modell, H, fenster)
            RES[(modell, H, fenster)] = res
            rows.append({"Modell": modell, "Fenster": fenster, **r})
    print(f"{modell} fertig ({time.time() - t0:.0f}s)")
ALL10 = pd.DataFrame(rows)
ALL10.to_csv(f"{OUT}/db10_ergebnisse.csv", index=False)

In [ ]:
# Zelle 5: Rangliste je Zeitraum. Durchschnitt über die Serien (die Serien sind NICHT unabhängig).
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
regeln = (ALL10[ALL10["Modell"] == "XGBoost"].groupby("Fenster")[["DA_NYMEX_Regel", "DA_Momentum"]].mean())
for fenster in FENSTER:
    g = ALL10[ALL10["Fenster"] == fenster].groupby("Modell").agg(
        Trefferquote=("DA", "mean"), Boot_lo_Mittel=("Boot_lo", "mean"),
        Serien_Boot_lo_ueber_50=("Boot_lo", lambda x: int((x > 0.5).sum())),
        Brier_Skill=("Brier_Skill", "mean"), Ersparnis_pro_Tag=("Ersparnis_vs_kaufen", "mean"),
        Serien_Ersparnis_ueber_0=("Ersp_lo", lambda x: int((x > 0).sum())))
    g = g.sort_values("Trefferquote", ascending=False)
    print(f"\n=== {fenster}: Rangliste (Mittel über {len(SERIEN_LAUF)} Serien) ===")
    print(g.round(3).to_string())
    print("Einfache Regeln, gleiche Tage: NYMEX-Regel", round(regeln.loc[fenster, "DA_NYMEX_Regel"], 3),
          "| Momentum", round(regeln.loc[fenster, "DA_Momentum"], 3))

In [ ]:
# Zelle 6: Ist ein Modell wirklich besser als XGBoost? Paarvergleich (McNemar) je Serie, Test-Zeitraum.
# Holm-Korrektur über alle Vergleiche. Erwartung: kaum etwas ist signifikant.
vgl = []
for modell in MODELLE:
    if modell == "XGBoost":
        continue
    for H in SERIEN_LAUF:
        a, b = RES[("XGBoost", H, "Test")], RES[(modell, H, "Test")]
        ok_x = ((a["proba"].values >= 0.5).astype(int) == a["y_true"].values)
        ok_m = ((b["proba"].values >= 0.5).astype(int) == b["y_true"].values)
        assert (a["date"].values == b["date"].values).all()
        vgl.append({"Modell": modell, "Serie": H, "DA_Modell": ok_m.mean(), "DA_XGBoost": ok_x.mean(),
                    "Differenz": ok_m.mean() - ok_x.mean(), "p_McNemar": mcnemar(ok_m, ok_x)})
V = pd.DataFrame(vgl)
V["p_holm"] = holm(V["p_McNemar"].values)
print(V.groupby("Modell").agg(Differenz_Mittel=("Differenz", "mean"), kleinster_p=("p_McNemar", "min"),
                              kleinster_p_holm=("p_holm", "min")).round(3).to_string())
print("\nVergleiche mit p_holm < 0.05:", int((V["p_holm"] < 0.05).sum()), "von", len(V))

In [ ]:
# Zelle 7: Einzelwerte je Serie für das beste Modell nach Entwicklungszeitraum (Auswahl OHNE Test-Blick).
dev_rang = ALL10[ALL10["Fenster"] == "Entwicklung"].groupby("Modell")["DA"].mean().sort_values(ascending=False)
bestes = dev_rang.index[0]
print("Bestes Modell nach Entwicklungszeitraum:", bestes, round(dev_rang.iloc[0], 3))
print("\nSein Ergebnis im Test-Zeitraum (hier erst angesehen):")
print(ALL10[(ALL10["Modell"] == bestes) & (ALL10["Fenster"] == "Test")][
    ["Serie", "n", "DA", "Wilson_lo", "Boot_lo", "Boot_hi", "p_nicht_ueberl", "DA_NYMEX_Regel", "Brier_Skill",
     "Ersparnis_vs_kaufen", "Ersp_lo"]].round(3).to_string(index=False))

## Interpretation (nach dem Lauf gemeinsam ausfüllen)
- Welches Modell liegt in **beiden** Zeiträumen vorn? Wenn die Reihenfolge wechselt, gibt es keinen klaren Sieger.
- Ist der Vorsprung gegenüber XGBoost und gegenüber der NYMEX-Regel größer als der Zufall? (Zelle 6, Holm)
- Was zeigt das Ergebnis **nicht**? Gilt nur für S0 (Kauf zum Preis von t), nur für diese Zeiträume, die 8 Serien sind abhängig.
- Auswahl des Modells erst nach dem Holdout-Test auf neuen Tagen.